In [1]:
import os

os.chdir(r"C:\Users\yoshi\Downloads\PandasAssignment")
print(os.getcwd())

C:\Users\yoshi\Downloads\PandasAssignment


In [2]:
#DATA INGESTION

#Importing the dataset and printing the first 5 rows using head func
import pandas as pd
df = pd.read_csv("household_power_consumption.txt", sep=";")

print(df.head())

C:\Users\yoshi\AppData\Local\Temp\ipykernel_22824\3024561332.py:5: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("household_power_consumption.txt", sep=";")


         Date      Time Global_active_power Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00               4.216                 0.418  234.840   
1  16/12/2006  17:25:00               5.360                 0.436  233.630   
2  16/12/2006  17:26:00               5.374                 0.498  233.290   
3  16/12/2006  17:27:00               5.388                 0.502  233.740   
4  16/12/2006  17:28:00               3.666                 0.528  235.680   

  Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
0           18.400          0.000          1.000            17.0  
1           23.000          0.000          1.000            16.0  
2           23.000          0.000          2.000            17.0  
3           23.000          0.000          1.000            17.0  
4           15.800          0.000          1.000            17.0  


In [3]:
#Adding timestamp 
#Initially only used DataFrame methods then used datetime func to turn into objects so python can undertsand it 
#Also learnt formatting here

df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"],format="%d/%m/%Y %H:%M:%S")
print(df.head())

         Date      Time Global_active_power Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00               4.216                 0.418  234.840   
1  16/12/2006  17:25:00               5.360                 0.436  233.630   
2  16/12/2006  17:26:00               5.374                 0.498  233.290   
3  16/12/2006  17:27:00               5.388                 0.502  233.740   
4  16/12/2006  17:28:00               3.666                 0.528  235.680   

  Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  \
0           18.400          0.000          1.000            17.0   
1           23.000          0.000          1.000            16.0   
2           23.000          0.000          2.000            17.0   
3           23.000          0.000          1.000            17.0   
4           15.800          0.000          1.000            17.0   

            timestamp  
0 2006-12-16 17:24:00  
1 2006-12-16 17:25:00  
2 2006-12-16 17:26:00  
3 2006-12-16 17:27:00  
4 

In [4]:
#before removing NULL vals, checking missing mixed values as per wrning initially
print((df == "?").sum())

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3               0
timestamp                    0
dtype: int64


In [5]:
#Checking data set size to verify after cleaning up
print(df.shape)

(2075259, 10)


In [6]:
df = df.replace("?", pd.NA) #turn missing to Null
df = df.dropna()

print(df.shape)  #check if we got rid of the null vals

(2049280, 10)


In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 2049280 entries, 0 to 2075258
Data columns (total 10 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Date                   object        
 1   Time                   object        
 2   Global_active_power    object        
 3   Global_reactive_power  object        
 4   Voltage                object        
 5   Global_intensity       object        
 6   Sub_metering_1         object        
 7   Sub_metering_2         object        
 8   Sub_metering_3         float64       
 9   timestamp              datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(8)
memory usage: 172.0+ MB
None


In [8]:
#Had to fix data types to run aggregate funcs more easily
df["Global_active_power"] = pd.to_numeric(df["Global_active_power"])
df["Global_reactive_power"] = pd.to_numeric(df["Global_reactive_power"])
df["Voltage"] = pd.to_numeric(df["Voltage"])
df["Global_intensity"] = pd.to_numeric(df["Global_intensity"])
df["Sub_metering_1"] = pd.to_numeric(df["Sub_metering_1"])
df["Sub_metering_2"] = pd.to_numeric(df["Sub_metering_2"])

In [9]:
#verifying the data types
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 2049280 entries, 0 to 2075258
Data columns (total 10 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Date                   object        
 1   Time                   object        
 2   Global_active_power    float64       
 3   Global_reactive_power  float64       
 4   Voltage                float64       
 5   Global_intensity       float64       
 6   Sub_metering_1         float64       
 7   Sub_metering_2         float64       
 8   Sub_metering_3         float64       
 9   timestamp              datetime64[ns]
dtypes: datetime64[ns](1), float64(7), object(2)
memory usage: 172.0+ MB
None


In [10]:
import sys
print(sys.executable)

C:\Users\yoshi\anaconda3\envs\ml_env\python.exe


In [11]:
import sys
!{sys.executable} -m pip install pymongo

In [12]:
import pymongo
print(pymongo.__version__)

4.17.0


In [13]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(client.list_database_names())

['admin', 'config', 'local', 'power_db']


In [14]:
db = client["power_db"]
collection = db["power_consumption"]
print(collection)

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'power_db'), 'power_consumption')


In [15]:
#new dataframe with 5 rows only rn
sample_df = df.head(5)

In [16]:
records = sample_df.to_dict("records")

In [17]:
#Convert to dictionary to be more compatibke to mongodb
records = sample_df.to_dict("records")

In [18]:
#Inserting into mongodb
result = collection.insert_many(records)

In [19]:
#Verifying if the insertion worked or nah
print(len(result.inserted_ids))

5


In [20]:
peak_row = df.loc[df["Global_active_power"].idxmax()]

print(peak_row["timestamp"])
print(peak_row["Global_active_power"])

2009-02-22 17:09:00
11.122


In [21]:
print(collection.count_documents({}))

2049290


In [22]:
#inserting the entire dataset/dataframe in batches
batch_size = 10000

for start in range(0, len(df), batch_size):
    batch = df.iloc[start:start + batch_size]
    records = batch.to_dict("records")
    collection.insert_many(records)
    print("Inserted:", start + len(batch))

Inserted: 10000
Inserted: 20000
Inserted: 30000
Inserted: 40000
Inserted: 50000
Inserted: 60000
Inserted: 70000
Inserted: 80000
Inserted: 90000
Inserted: 100000
Inserted: 110000
Inserted: 120000
Inserted: 130000
Inserted: 140000
Inserted: 150000
Inserted: 160000
Inserted: 170000
Inserted: 180000
Inserted: 190000
Inserted: 200000
Inserted: 210000
Inserted: 220000
Inserted: 230000
Inserted: 240000
Inserted: 250000
Inserted: 260000
Inserted: 270000
Inserted: 280000
Inserted: 290000
Inserted: 300000
Inserted: 310000
Inserted: 320000
Inserted: 330000
Inserted: 340000
Inserted: 350000
Inserted: 360000
Inserted: 370000
Inserted: 380000
Inserted: 390000
Inserted: 400000
Inserted: 410000
Inserted: 420000
Inserted: 430000
Inserted: 440000
Inserted: 450000
Inserted: 460000
Inserted: 470000
Inserted: 480000
Inserted: 490000
Inserted: 500000
Inserted: 510000
Inserted: 520000
Inserted: 530000
Inserted: 540000
Inserted: 550000
Inserted: 560000
Inserted: 570000
Inserted: 580000
Inserted: 590000
Insert

In [23]:
#checking insertion succes
print(collection.count_documents({}))

4098570


In [24]:
#PART 2
#finding the peak consumption
peak = collection.find_one(
    sort=[("Global_active_power", -1)]
)

print("Timestamp:", peak["timestamp"])
print("Power:", peak["Global_active_power"])

Timestamp: 2009-02-22 17:09:00
Power: 11.122


In [25]:
#fetch all data points for February 2nd, 2008.
from datetime import datetime

start = datetime(2008, 2, 2)
end = datetime(2008, 2, 3)
records = list(
    collection.find(
        {
            "timestamp": {
                "$gte": start,
                "$lt": end
            }
        }
    )
)

print("Records found:", len(records))

#Since the original dataset contained missing values were removed , 
#the count is slightly lower than the USUAL 1440 minute-level measurements for a  day.

Records found: 2878


In [26]:
#Finding records where global_active_power>7, using query by val
records = list(
    collection.find(
        {
            "Global_active_power": {
                "$gt": 7
            }
        }
    )
)
print("Count:", len(records))

Count: 3238


In [27]:
#Basic statistics
pipeline = [
    {
        "$group": {
            "_id": None,

            "average_power": {
                "$avg": "$Global_active_power"
            },

            "minimum_power": {
                "$min": "$Global_active_power"
            },

            "maximum_power": {
                "$max": "$Global_active_power"
            }
        }
    }
]

result = list(collection.aggregate(pipeline))
print(result)

[{'_id': None, 'average_power': 1.091624086449664, 'minimum_power': 0.076, 'maximum_power': 11.122}]


In [28]:
#Monthly consumption since of each month in 2008
from datetime import datetime

pipeline = [

    {
        "$match": {
            "timestamp": {
                "$gte": datetime(2008, 1, 1),
                "$lt": datetime(2009, 1, 1)
            }
        }
    },

    {
        "$group": {
            "_id": {
                "$month": "$timestamp"
            },

            "total_power": {
                "$sum": "$Global_active_power"
            }
        }
    },

    {
        "$sort": {
            "_id": 1
        }
    }

]

result = list(collection.aggregate(pipeline))

for month in result:
    print(month)

{'_id': 1, 'total_power': 130338.764}
{'_id': 2, 'total_power': 98662.144}
{'_id': 3, 'total_power': 111181.188}
{'_id': 4, 'total_power': 96419.984}
{'_id': 5, 'total_power': 91443.72}
{'_id': 6, 'total_power': 85887.944}
{'_id': 7, 'total_power': 70954.836}
{'_id': 8, 'total_power': 24683.76}
{'_id': 9, 'total_power': 85335.584}
{'_id': 10, 'total_power': 101392.92}
{'_id': 11, 'total_power': 119817.52}
{'_id': 12, 'total_power': 113667.824}


In [29]:
import os

print(os.getcwd())

C:\Users\yoshi\Downloads\PandasAssignment
